In [6]:
from __future__ import annotations
import re
from pathlib import Path
import os
import sys
import pandas as pd
import numpy as np

## [1] data 확인

- `2505` 등의 날짜 형태만 바꿔서 쓰면 됨

In [18]:
data_dir = Path(r"\\DocuONE\MyDrive\개인함\전년_전월대비_계량기\data")
export_dir = Path(r"C:\DA\exports")
export_dir.mkdir(exist_ok=True)

In [ ]:
file_2505 = data_dir / "계량기_2505.xlsx"
file_2604 = data_dir / "계량기_2604.xlsx"
file_2605 = data_dir / "계량기_2605.xlsx"

In [21]:
try:
    ALL_2505 = pd.read_excel(file_2505, sheet_name='자원리스트')
    print(f"✔ 2505월 완료: {ALL_2505.shape[0]:,}행 로드")
    
    ALL_2604 = pd.read_excel(file_2604, sheet_name='자원리스트')
    print(f"✔ 2604월 완료: {ALL_2604.shape[0]:,}행 로드")
    
    ALL_2605 = pd.read_excel(file_2605, sheet_name='자원리스트')
    print(f"✔ 2605월 완료: {ALL_2605.shape[0]:,}행 로드")
except Exception as e:
    print(f"❌ 로드 실패! 에러 내용: {e}")

✔ 25년 5월 완료: 835,578행 로드
✔ 26년 4월 완료: 842,738행 로드
✔ 26년 5월 완료: 842,790행 로드


In [24]:
ALL_2605.head()

,고객센터,시/구/군,건물번호,단지번호,세대번호,세대유형,세대유형텍스트,평수,설치번호,설치유형,...,압력(바),TYPE,계량기유형,계량기유형명,설치번호생성일,동,전입일자,전출일자,특정구분,인덕션여부
0,H051,계룡시,1083314,H2500577,1048745,110,아파트,18,1039451,2,...,0.0294,좌타입,2900.0,일반원격-원격연결 무,20020821,엄사면,20150614,99991231,비특정,연소기
1,H051,계룡시,1083314,H2500577,1048751,110,아파트,18,1039467,2,...,0.0294,좌타입,2900.0,일반원격-원격연결 무,20020821,엄사면,20140920,20260322,비특정,인덕션
2,H051,계룡시,1083314,H2500577,1048760,110,아파트,18,1039477,2,...,0.0294,좌타입,2900.0,일반원격-원격연결 무,20020821,엄사면,20151204,99991231,비특정,인덕션
3,H051,계룡시,1083314,H2500577,1048771,110,아파트,18,1039491,2,...,0.0294,좌타입,2900.0,일반원격-원격연결 무,20020821,엄사면,20061124,99991231,비특정,인덕션
4,H051,계룡시,1083314,H2500577,1048796,110,아파트,18,1039503,2,...,0.0294,좌타입,2900.0,일반원격-원격연결 무,20020821,엄사면,20020816,99991231,비특정,연소기


> 조건 적용
- 설치유형명에서 제외 [가정용 난방, 가정용 취사, 일반용1]
- 계량기번호 = 첫 두 글자 'HM'

In [35]:
exclude_types = ['가정용 난방', '가정용 취사', '일반용1']
def make_filtered(df):
    col_type = "설치유형명" if "설치유형명" in df.columns else df.columns[10]
    col_meter = "계량기번호"
    col_induction = "인덕션여부"  # 인덕션여부 컬럼
    
    return df[
        # 1. exclude_types에 포함되지 않는 것
        (~df[col_type].isin(exclude_types)) &
        
        # 2. 계량기번호가 'HM'으로 시작하는 것
        (df[col_meter].fillna('').astype(str).str.startswith('HM')) 
        
    ].copy()
    
print("\n가정용 제외 및 인덕션여부 값이 있는 계량기 필터링 진행 중...")
SUM_2505 = make_filtered(ALL_2505)
SUM_2604 = make_filtered(ALL_2604)
SUM_2605 = make_filtered(ALL_2605)
print(f"필터링 완료 -> 2505: {SUM_2505.shape[0]:,}개 | 2604: {SUM_2604.shape[0]:,}개 | 2605: {SUM_2605.shape[0]:,}개")


가정용 제외 및 인덕션여부 값이 있는 계량기 필터링 진행 중...
필터링 완료 -> 2505: 10,936개 | 2604: 10,932개 | 2605: 10,929개


## [2] 사업부별 계량기 집계

### (1) 전입(생성)/전출(삭제) 행

In [27]:
def get_changed_rows(prev_df, curr_df, key_col='설치번호'):
    prev = prev_df.copy()
    curr = curr_df.copy()

    prev[key_col] = prev[key_col].astype(str)
    curr[key_col] = curr[key_col].astype(str)

    prev_keys = set(prev[key_col])
    curr_keys = set(curr[key_col])

    added_keys = curr_keys - prev_keys
    removed_keys = prev_keys - curr_keys

    added_rows = curr[curr[key_col].isin(added_keys)].sort_values(['고객센터', key_col]).reset_index(drop=True)
    removed_rows = prev[prev[key_col].isin(removed_keys)].sort_values(['고객센터', key_col]).reset_index(drop=True)
    # 불필요한 Unnamed 컬럼 정리

    cols_to_drop = [c for c in added_rows.columns if "Unnamed" in c]
    if cols_to_drop:
        added_rows = added_rows.drop(columns=cols_to_drop)
        removed_rows = removed_rows.drop(columns=cols_to_drop)
    return added_rows, removed_rows

In [31]:
# (1) 전월 대비 (26년 4월 -> 26년 5월)
ADDED_2605_FROM_2604, REMOVED_2605_FROM_2604 = get_changed_rows(SUM_2604, SUM_2605)
# (2) 전년 대비 (25년 5월 -> 26년 5월)
ADDED_2605_FROM_2505, REMOVED_2605_FROM_2505 = get_changed_rows(SUM_2505, SUM_2605)
ADDED_2605_FROM_2505

,고객센터,시/구/군,건물번호,단지번호,세대번호,세대유형,세대유형텍스트,평수,설치번호,설치유형,...,압력(바),TYPE,계량기유형,계량기유형명,설치번호생성일,동,전입일자,전출일자,특정구분,인덕션여부
0,H051,계룡시,2938135,H2500233,2607697,730,사회복지시설(노인정포함),NaN,2672420,6,...,0.0294,좌타입,2900.0,일반원격-원격연결 무,20110421,엄사면,20240229,99991231,비특정,NaN
1,H051,계룡시,5053471,H2500899,4520205,730,사회복지시설(노인정포함),NaN,4586664,10,...,0.0294,우타입,2900.0,일반원격-원격연결 무,20250704,엄사면,20250529,99991231,특정(시도고시),NaN
2,H051,계룡시,5053471,H2500899,4520204,730,사회복지시설(노인정포함),NaN,4586665,13,...,0.0294,우타입,2900.0,일반원격-원격연결 무,20250704,엄사면,20250529,99991231,특정(시도고시),NaN
3,H051,계룡시,4791186,H2500582,4522578,1099,기타,NaN,4589849,8,...,0.0294,우타입,1900.0,일반-원격연결 무,20251031,두마면,20251029,99991231,특정(시도고시),NaN
4,H051,계룡시,1093718,H2500222,4526167,830,"일반사무(판매,제조이외)",NaN,4590745,6,...,0.0294,좌타입,1900.0,일반-원격연결 무,20251222,엄사면,20251222,99991231,비특정,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
131,H075,유성구,5062219,H2011938,4527297,1099,기타,NaN,4594571,6,...,0.0294,우타입,1900.0,일반-원격연결 무,20260507,용계동,20260506,99991231,특정(1종보호),NaN
132,H075,유성구,5062219,H2011938,4527296,1099,기타,NaN,4594572,6,...,0.0294,우타입,1900.0,일반-원격연결 무,20260507,용계동,20260506,99991231,특정(1종보호),NaN
133,H075,유성구,5062221,H2011938,4527303,1099,기타,NaN,4594577,6,...,0.0294,우타입,1900.0,일반-원격연결 무,20260507,용계동,20260506,99991231,특정(1종보호),NaN
134,H075,유성구,5062331,H2011942,4527375,830,"일반사무(판매,제조이외)",NaN,4594635,6,...,0.0294,우타입,1900.0,일반-원격연결 무,20260527,구암동,20260528,99991231,비특정,NaN


### (2) 사업부 월별 계량기 집계

In [36]:
cnt_2505 = SUM_2505['고객센터'].value_counts().rename('2505')
cnt_2604 = SUM_2604['고객센터'].value_counts().rename('2604')
cnt_2605 = SUM_2605['고객센터'].value_counts().rename('2605')

In [40]:
monthly_counts = pd.concat([cnt_2505, cnt_2604, cnt_2605], axis=1).fillna(0).astype(int)
monthly_counts.index.name = '고객센터'
monthly_counts = monthly_counts.reset_index().sort_values('고객센터').reset_index(drop=True)

summary_row = pd.DataFrame([{
    '고객센터': '합계',
    '2505': monthly_counts['2505'].sum(),
    '2604': monthly_counts['2604'].sum(),
    '2605': monthly_counts['2605'].sum()
}])
monthly_counts = pd.concat([monthly_counts, summary_row], ignore_index=True)
monthly_counts

,고객센터,2505,2604,2605
0,H051,338,341,342
1,H071,2393,2358,2358
2,H072,2496,2508,2505
3,H073,1769,1772,1770
4,H074,1943,1951,1951
5,H075,1997,2002,2003
6,합계,10936,10932,10929


### (3) 증감표

In [48]:
added_monthly_cnt = ADDED_2605_FROM_2604['고객센터'].value_counts()
removed_monthly_cnt = REMOVED_2605_FROM_2604['고객센터'].value_counts()
added_yearly_cnt = ADDED_2605_FROM_2505['고객센터'].value_counts()
removed_yearly_cnt = REMOVED_2605_FROM_2505['고객센터'].value_counts()

In [50]:
centers = monthly_counts[monthly_counts['고객센터'] != '합계']['고객센터']
change_compare = pd.DataFrame()
change_compare['고객센터'] = centers

In [ ]:
change_compare['2505'] = change_compare['고객센터'].map(monthly_counts.set_index('고객센터')['2505'])
change_compare['2604'] = change_compare['고객센터'].map(monthly_counts.set_index('고객센터')['2604'])
change_compare['2605'] = change_compare['고객센터'].map(monthly_counts.set_index('고객센터')['2605'])
 

In [54]:
# 전년대비 (2505 -> 2605) [증가 / 감소 / 증감] 집계
change_compare['전년대비 증가'] = change_compare['고객센터'].map(added_yearly_cnt).fillna(0).astype(int)
change_compare['전년대비 감소'] = change_compare['고객센터'].map(removed_yearly_cnt).fillna(0).astype(int)
change_compare['전년대비 증감'] = change_compare['전년대비 증가'] - change_compare['전년대비 감소']  # 순증감
# 전월대비 (2604 -> 2605) [증가 / 감소 / 증감] 집계
change_compare['전월대비 증가'] = change_compare['고객센터'].map(added_monthly_cnt).fillna(0).astype(int)
change_compare['전월대비 감소'] = change_compare['고객센터'].map(removed_monthly_cnt).fillna(0).astype(int)
change_compare['전월대비 증감'] = change_compare['전월대비 증가'] - change_compare['전월대비 감소']  # 순증감

In [55]:
sum_row = pd.DataFrame([{
    '고객센터': '합계',
    '2505': change_compare['2505'].sum(),
    '2604': change_compare['2604'].sum(),
    '2605': change_compare['2605'].sum(),
    '전년대비 증가': change_compare['전년대비 증가'].sum(),
    '전년대비 감소': change_compare['전년대비 감소'].sum(),
    '전년대비 증감': change_compare['전년대비 증감'].sum(),
    '전월대비 증가': change_compare['전월대비 증가'].sum(),
    '전월대비 감소': change_compare['전월대비 감소'].sum(),
    '전월대비 증감': change_compare['전월대비 증감'].sum()
}])
change_compare = pd.concat([change_compare, sum_row], ignore_index=True)
# 결과 확인
change_compare

,고객센터,2505,2604,2605,전년대비 증가,전년대비 감소,전년대비 증감,전월대비 증가,전월대비 감소,전월대비 증감
0,H051,338,341,342,6,2,4,1,0,1
1,H071,2393,2358,2358,6,41,-35,1,1,0
2,H072,2496,2508,2505,38,29,9,1,4,-3
3,H073,1769,1772,1770,17,16,1,0,2,-2
4,H074,1943,1951,1951,40,32,8,2,2,0
5,H075,1997,2002,2003,29,23,6,7,6,1
6,합계,10936,10932,10929,136,143,-7,12,15,-3


## [3] 저장

In [56]:
output_excel = export_dir / "202605_계량기증감.xlsx"
with pd.ExcelWriter(output_excel, engine='openpyxl') as writer:
    # [시트 1] 월별 사업부별 계량기 건수 (합계 포함)
    monthly_counts.to_excel(writer, sheet_name="월별_사업부별_건수", index=False)
    
    # [시트 2] 2605 대비 전년/전월 증감 비교표 (합계 포함)
    change_compare.to_excel(writer, sheet_name="증감_비교", index=False)
    
    # [시트 3] 전월 대비 신규 생성 리스트 (전체 행)
    ADDED_2605_FROM_2604.to_excel(writer, sheet_name="전월_생성", index=False)
    
    # [시트 4] 전월 대비 삭제 리스트 (전체 행)
    REMOVED_2605_FROM_2604.to_excel(writer, sheet_name="전월_삭제", index=False)
    
    # [시트 5] 전년 대비 신규 생성 리스트 (전체 행)
    ADDED_2605_FROM_2505.to_excel(writer, sheet_name="전년_생성", index=False)
    
    # [시트 6] 전년 대비 삭제 리스트 (전체 행)
    REMOVED_2605_FROM_2505.to_excel(writer, sheet_name="전년_삭제", index=False)